### Understanding Ridge Regression for Multicollinearity

This notebook demonstrates a key problem in linear regression called **multicollinearity** and how **Ridge Regression** provides a solution.

**The Problem (Multicollinearity):**
When two or more predictor variables (like `x1` and `x2` in this example) are highly correlated, the standard linear regression equation becomes unstable. This instability is reflected in a very small (near zero) determinant of the `X.T @ X` matrix, making it difficult to calculate the inverse needed for finding the regression coefficients. This leads to wildly fluctuating and unreliable coefficient estimates.

In our example, `x2` is almost a perfect multiple of `x1`, creating extreme multicollinearity.

**The Solution (Ridge Regression):**
Ridge Regression addresses multicollinearity by adding a small 'penalty' (controlled by `lambda_val`) to the diagonal elements of the `X.T @ X` matrix. This penalty makes the matrix invertible and stabilizes the coefficient estimates. The key steps are:

1.  **Generate data:** Create highly multicollinear data for demonstration.
2.  **Set `lambda_val`:** A penalty term (here, `1.0`) is chosen. Even a small value can significantly improve stability.
3.  **Create Identity Matrix `I`:** An identity matrix is used for the penalty. Importantly, the penalty is *not* applied to the intercept term (`I[0,0] = 0`), as we only want to shrink the slopes of the features, not the baseline.
4.  **Apply Ridge Normal Equation:** The term `(lambda_val * I)` is added to `X.T @ X` to form `Ridge_Matrix`. This modification increases the determinant of the matrix, making it well-conditioned for inversion.
5.  **Calculate Stable Slopes:** The `beta_ridge` values are then calculated using the inverse of this stabilized `Ridge_Matrix`.

**Results:**
By applying Ridge Regression, the original unstable model (which would have yielded nonsensical slopes due to multicollinearity) is stabilized. The `new_determinant` is now a substantial positive value, confirming the matrix is well-conditioned. The `beta_ridge` values show reasonable and interpretable slopes, effectively handling the multicollinearity between `x1` and `x2`.

In [1]:
import numpy as np

np.set_printoptions(suppress=True, precision=6)

# 1. Generate the exact same broken, multicollinear data
np.random.seed(42)
x1 = np.random.normal(loc=0, scale=1, size=100)
x2 = (x1 * 2) + np.random.normal(loc=0, scale=0.00000001, size=100)
y = 10 + (5 * x1) + np.random.normal(loc=0, scale=1, size=100)

ones = np.ones(100)
X = np.column_stack((ones, x1, x2))

# 2. Set our Ridge Penalty (Lambda)
lambda_val = 1.0  # Even a small penalty fixes the math!

# 3. Create the Identity Matrix (I)
# It needs to be the same size as the number of features (3x3)
I = np.eye(X.shape[1])

# CRITICAL RULE: We do NOT penalize the y-intercept (Beta_0)!
# We only want to shrink the slopes, not the starting baseline of the data.
I[0, 0] = 0

# 4. The Ridge Normal Equation
X_T_X = X.T @ X
Ridge_Matrix = X_T_X + (lambda_val * I)

# Let's check the new determinant!
new_determinant = np.linalg.det(Ridge_Matrix)
print(f"Old broken determinant was ~0.0000")
print(f"New Ridge Determinant: {new_determinant:.2f}\n")

# Calculate the new stable slopes
beta_ridge = np.linalg.inv(Ridge_Matrix) @ X.T @ y

print("--- The Stabilized Slopes ---")
feature_names = ["Intercept", "x1 (The real feature)", "x2 (The duplicate)"]
for name, b in zip(feature_names, beta_ridge):
    print(f"{name:25} | Ridge Slope: {b:10.4f}")

Old broken determinant was ~0.0000
New Ridge Determinant: 40926.11

--- The Stabilized Slopes ---
Intercept                 | Ridge Slope:    10.0872
x1 (The real feature)     | Ridge Slope:     1.0430
x2 (The duplicate)        | Ridge Slope:     2.0860
